# 🌾 RAG Pipeline: AgroLLM PDF Knowledge Base → FAISS Index

**Dataset:** [`viswambhar/agrollm-data`](https://www.kaggle.com/datasets/viswambhar/agrollm-data)

## Dataset Structure (31 PDFs across 4 domain folders)
```
agrollm_dataset/
├── Agri_life_sciences/          (5 PDFs)
├── Agricultural_management/     (8 PDFs)
├── Agriculture_and_forestry/    (3 PDFs)
└── Agriculture_business/        (15 PDFs)
```

## Pipeline
1. Extract text from all PDFs using `PyMuPDF`
2. Chunk text into 300-word overlapping pieces
3. Embed with multilingual sentence-transformers (EN + AR)
4. Build FAISS vector index
5. Push index to Hugging Face Hub for the Gradio app

## 1 - Install Dependencies

In [ ]:
%%capture
!pip install -q kaggle pymupdf sentence-transformers faiss-cpu huggingface_hub numpy

## 2 - Download Dataset from Kaggle

In [ ]:
import os
import json

# ── Paste your Kaggle API credentials ───────────────────────────────────────
KAGGLE_USERNAME = "your_kaggle_username"   # ← replace
KAGGLE_KEY      = "your_kaggle_api_key"    # ← replace

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Kaggle credentials set")

In [ ]:
!kaggle datasets download -d viswambhar/agrollm-data --unzip -p ./agrollm_data

# List all PDFs found
import glob
pdf_files = sorted(glob.glob("./agrollm_data/**/*.pdf", recursive=True))
print(f"\n✅ Found {len(pdf_files)} PDF files:\n")
for p in pdf_files:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {p.replace('./agrollm_data/', '')} ({size_mb:.1f} MB)")

## 3 - Extract Text from All PDFs

In [ ]:
import fitz   # PyMuPDF
import re
import time

def extract_pdf_text(pdf_path):
    """
    Extract clean text from a PDF file using PyMuPDF.
    Handles multi-column layouts and cleans up artifacts.
    """
    doc = fitz.open(pdf_path)
    full_text = []
    for page in doc:
        text = page.get_text("text")   # plain text extraction
        # Clean up common PDF artifacts
        text = re.sub(r'\s+', ' ', text)          # collapse whitespace
        text = re.sub(r'-\s+', '', text)           # fix hyphenated line breaks
        text = re.sub(r'\x00', '', text)           # remove null bytes
        text = text.strip()
        if len(text) > 50:                          # skip mostly-empty pages
            full_text.append(text)
    doc.close()
    return ' '.join(full_text)


def get_domain(pdf_path):
    """Extract domain folder name from PDF path."""
    parts = pdf_path.replace("\\", "/").split("/")
    for part in parts:
        if part in ['Agri_life_sciences', 'Agricultural_management',
                    'Agriculture_and_forestry', 'Agriculture_business']:
            return part.replace("_", " ")
    return "Agriculture"


# Extract text from all PDFs
print(f"⏳ Extracting text from {len(pdf_files)} PDFs...\n")
t0 = time.time()

raw_documents = []
failed = []

for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    domain = get_domain(pdf_path)
    try:
        text = extract_pdf_text(pdf_path)
        word_count = len(text.split())
        if word_count > 100:
            raw_documents.append({
                "text": text,
                "source": fname,
                "domain": domain,
                "word_count": word_count
            })
            print(f"  ✅ {fname}: {word_count:,} words")
        else:
            print(f"  ⚠️ {fname}: only {word_count} words (skipped — possibly scanned)")
            failed.append(fname)
    except Exception as e:
        print(f"  ❌ {fname}: {e}")
        failed.append(fname)

total_words = sum(d['word_count'] for d in raw_documents)
print(f"\n✅ Successfully extracted: {len(raw_documents)}/{len(pdf_files)} PDFs")
print(f"   Total words: {total_words:,}")
print(f"   Extraction time: {time.time()-t0:.1f}s")
if failed:
    print(f"\n⚠️ Failed/skipped: {failed}")

In [ ]:
# Preview extracted content
for doc in raw_documents[:3]:
    print(f"\n{'='*60}")
    print(f"📄 {doc['source']} [{doc['domain']}] — {doc['word_count']:,} words")
    print(f"Preview: {doc['text'][:300]}...")

## 4 - Chunk Documents

In [ ]:
def chunk_document(doc, chunk_size=300, overlap=50):
    """
    Split a document into overlapping word-level chunks.
    Each chunk inherits metadata (source, domain).
    """
    words = doc['text'].split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_text = ' '.join(words[start:end]).strip()
        if len(chunk_text) > 100:  # skip too-short chunks
            chunks.append({
                "text": chunk_text,
                "source": doc['source'],
                "domain": doc['domain'],
            })
        start += chunk_size - overlap   # overlap between chunks
    return chunks


all_chunks = []
for doc in raw_documents:
    chunks = chunk_document(doc, chunk_size=300, overlap=50)
    all_chunks.extend(chunks)
    print(f"  {doc['source']}: {doc['word_count']:,} words → {len(chunks)} chunks")

print(f"\n✅ Total chunks: {len(all_chunks):,}")
lens = [len(c['text'].split()) for c in all_chunks]
print(f"   Chunk word count — min: {min(lens)}, max: {max(lens)}, avg: {sum(lens)//len(lens)}")

# Domain distribution
from collections import Counter
domain_counts = Counter(c['domain'] for c in all_chunks)
print("\n📊 Chunks per domain:")
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count:,} chunks")

## 5 - Embed Chunks

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"⬇️  Loading: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)
print("✅ Embedder ready")

texts = [c['text'] for c in all_chunks]
print(f"\n⏳ Embedding {len(texts):,} chunks (this takes a few minutes)...")
t0 = time.time()

embeddings = embedder.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"\n✅ Embeddings: {embeddings.shape}")
print(f"   Time: {(time.time()-t0)/60:.1f} minutes")

## 6 - Build FAISS Index

In [ ]:
import faiss

EMBED_DIM = embeddings.shape[1]
index = faiss.IndexFlatIP(EMBED_DIM)   # Inner product = cosine similarity (normalized)
index.add(embeddings.astype(np.float32))

print(f"✅ FAISS index built")
print(f"   Vectors: {index.ntotal:,}")
print(f"   Dimension: {EMBED_DIM}")

# ── Quick tests ───────────────────────────────────────────────────────────────
test_queries = [
    "What are symptoms of powdery mildew?",
    "How to improve soil fertility?",
    "Best practices for drip irrigation",
    "ما هي أمراض القمح؟",   # Arabic: What are wheat diseases?
]

for query in test_queries:
    q_embed = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    dists, idxs = index.search(q_embed, k=2)
    print(f"\n🔍 Query: '{query}'")
    for dist, idx in zip(dists[0], idxs[0]):
        chunk = all_chunks[idx]
        print(f"  [{chunk['domain']} | {chunk['source']} | score={dist:.3f}]")
        print(f"  {chunk['text'][:120]}...")

## 7 - Save Index & Push to Hugging Face Hub

In [ ]:
from huggingface_hub import login, HfApi

SAVE_DIR = "./agro_rag_index"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save FAISS index
faiss.write_index(index, f"{SAVE_DIR}/agro.index")

# Save chunks (with metadata)
with open(f"{SAVE_DIR}/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

# Save config
config = {
    "embed_model": EMBED_MODEL,
    "embed_dim": EMBED_DIM,
    "num_chunks": len(all_chunks),
    "chunk_size_words": 300,
    "overlap_words": 50,
    "top_k": 3,
    "num_pdfs": len(raw_documents),
    "domains": list(domain_counts.keys()),
}
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("📁 Saved files:")
for fname in os.listdir(SAVE_DIR):
    size = os.path.getsize(f"{SAVE_DIR}/{fname}") / 1e6
    print(f"   {fname}: {size:.1f} MB")

# ── Push to Hub ────────────────────────────────────────────────────────────
login()

HF_USERNAME = "Rady10"
RAG_REPO    = f"{HF_USERNAME}/Plant-Disease-AgroRAG-Index"

api = HfApi()
try:
    api.repo_info(RAG_REPO, repo_type="dataset")
    print(f"\n✅ Repo exists: {RAG_REPO}")
except Exception:
    api.create_repo(RAG_REPO, repo_type="dataset", private=False)
    print(f"\n✅ Created repo: {RAG_REPO}")

api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=RAG_REPO,
    repo_type="dataset",
    commit_message=f"AgroLLM FAISS index: {len(raw_documents)} PDFs, {len(all_chunks):,} chunks",
)

print(f"\n✅ RAG index live at: https://huggingface.co/datasets/{RAG_REPO}")
print(f"\nUse in app.py:")
print(f'   RAG_REPO = "{RAG_REPO}"')

## Summary

| Item | Value |
|---|---|
| **PDFs processed** | 31 PDFs across 4 domain folders |
| **Domains** | Agri Life Sciences, Agricultural Management, Agriculture & Forestry, Agriculture Business |
| **Embedding model** | `paraphrase-multilingual-MiniLM-L12-v2` (EN + AR) |
| **Chunk size** | 300 words, 50-word overlap |
| **Vector store** | FAISS flat inner product (cosine similarity) |
| **RAG index repo** | `Rady10/Plant-Disease-AgroRAG-Index` |

> ⚡ The `app.py` already includes the full RAG pipeline — just run this notebook first to build and upload the index.